# S10 – Unidad 1. Evidencia de Aprendizaje 1
## Carga y creación de dataset

**Objetivo:** cargar datos desde Hugging Face, transformarlos con Pandas y generar un resumen del DataFrame transformado en un archivo `.txt`.

Este notebook coordina las clases `HFDatasetLoader` y `DatasetSummarizer`, que se encuentran en los archivos `.py` de esta carpeta.


## 1. Importar librerías y clases

Verifica que `dataset_loader.py` y `dataset_summarizer.py` estén en la misma carpeta que este notebook.


In [ ]:
import pandas as pd
from dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer


## 2. Cargar el dataset desde Hugging Face

Se usa `scikit-learn/iris`, mencionado como ejemplo en las instrucciones de la actividad. La primera ejecución requiere conexión a Internet.


In [ ]:
loader = HFDatasetLoader(dataset_id="scikit-learn/iris", split="train")
df_original = loader.load_data()

print("Dimensiones originales:", df_original.shape)
display(df_original.head())
print("\\nColumnas:", list(df_original.columns))


## 3. Transformación 1: renombrar una columna

Se renombra la primera columna disponible para demostrar una transformación explícita. El resto de las columnas se conserva.


In [ ]:
df_trabajo = df_original.copy()

primera_columna = df_trabajo.columns[0]
nombre_nuevo = "variable_renombrada"
# Evita conflicto si el dataset ya tiene una columna con ese nombre.
if nombre_nuevo in df_trabajo.columns and primera_columna != nombre_nuevo:
    nombre_nuevo = "variable_renombrada_1"

df_trabajo = df_trabajo.rename(columns={primera_columna: nombre_nuevo})

print("Columna renombrada:", primera_columna, "->", nombre_nuevo)
display(df_trabajo.head())


## 4. Transformación 2: seleccionar las columnas de trabajo

Se crea una nueva tabla con las columnas disponibles. Esto deja explícita la operación de selección sin depender de nombres específicos del dataset.


In [ ]:
columnas_seleccionadas = list(df_trabajo.columns)
df_trabajo = df_trabajo.loc[:, columnas_seleccionadas].copy()

print("Columnas seleccionadas:", list(df_trabajo.columns))
display(df_trabajo.head())


## 5. Transformación 3: tratar valores nulos numéricos

En las columnas numéricas, los valores faltantes se reemplazan por la mediana de su columna. Si una columna no tiene valores nulos, sus valores permanecen iguales.


In [ ]:
columnas_numericas = df_trabajo.select_dtypes(include="number").columns

for columna in columnas_numericas:
    if df_trabajo[columna].isna().any():
        mediana = df_trabajo[columna].median()
        df_trabajo[columna] = df_trabajo[columna].fillna(mediana)

print("Columnas numéricas revisadas:", list(columnas_numericas))
print("Valores nulos después del tratamiento:")
display(df_trabajo.isna().sum().to_frame(name="valores_nulos"))


## 6. Transformación 4: filtrar filas válidas

Se conservan las filas que tienen un valor no nulo en la primera columna de la tabla transformada. Se informa cuántas filas quedan después del filtro.


In [ ]:
columna_filtro = df_trabajo.columns[0]
filas_antes = len(df_trabajo)

df_trabajo = df_trabajo.loc[df_trabajo[columna_filtro].notna()].copy()

print("Columna usada para filtrar:", columna_filtro)
print("Filas antes del filtro:", filas_antes)
print("Filas después del filtro:", len(df_trabajo))
display(df_trabajo.head())


## 7. Generar el resumen TXT del DataFrame transformado

La clase recibe el DataFrame después de las cuatro transformaciones. El archivo generado corresponde a estos datos transformados, no al DataFrame original.


In [ ]:
resumidor = DatasetSummarizer(df_trabajo)
archivo_resumen = resumidor.generate_summary("resumen_dataset.txt")

print("Resumen generado:", archivo_resumen)

with open(archivo_resumen, "r", encoding="utf-8") as archivo:
    print(archivo.read())


## 8. Comprobación final

La siguiente celda verifica las dimensiones finales y que el DataFrame transformado esté disponible.


In [ ]:
print("Proceso finalizado correctamente.")
print(f"DataFrame final: {df_trabajo.shape[0]} filas y {df_trabajo.shape[1]} columnas.")
print("Archivo de resumen:", archivo_resumen)


## Conclusión

En esta actividad se implementaron dos clases: una para cargar datos desde Hugging Face y convertirlos en un DataFrame de Pandas, y otra para describir el DataFrame y exportar el resultado a un archivo de texto. El notebook coordina ambas clases y aplica cuatro transformaciones: renombrar una columna, seleccionar columnas, tratar valores nulos numéricos y filtrar filas con un valor presente en la primera columna.

El resumen TXT se genera a partir del DataFrame transformado. Antes de entregar, se debe ejecutar el notebook completo, revisar los resultados y publicar los archivos junto con los commits en GitHub.
